# Student Final Mathematics Grade Prediction Using Linear Regression

**Mini-project | Python | Google Colab / Jupyter Notebook**

## Project goal
Build a supervised regression model that estimates a student's final Mathematics grade (`G3`, scored from 0 to 20) using demographic, family, school, lifestyle and study-related information.

**Important design decision:** `G1` and `G2` are deliberately excluded from the predictors. They are earlier Mathematics grades and are strongly related to `G3`; using them would make the task easier but would no longer test prediction from non-grade characteristics.

**Dataset:** UCI Student Performance, Mathematics file `student-mat.csv` (395 records).

> The dataset represents students from two Portuguese schools. Results should not be assumed to generalize to other schools or countries.

## How to use this notebook

1. Open this notebook in Google Colab or Jupyter.
2. Run every cell from top to bottom.
3. Keep an internet connection for the first dataset download.
4. Read the printed outputs and charts; do not copy assumed scores into your report.
5. The notebook saves charts in `plots/` and metrics, predictions and coefficients in `results/`.

## 1. Install and import the required libraries

The project uses pandas and NumPy for data handling, Matplotlib and Seaborn for plots, and scikit-learn for preprocessing, model training and evaluation.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn joblib requests

In [ ]:
from pathlib import Path
from io import BytesIO
import zipfile
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "G3"

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "raw"
PLOTS_DIR = PROJECT_DIR / "plots"
RESULTS_DIR = PROJECT_DIR / "results"
for folder in (DATA_DIR, PLOTS_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
print("Project folder:", PROJECT_DIR.resolve())

## 2. Download and load the Mathematics dataset

The UCI archive contains separate files for Mathematics (`student-mat.csv`) and Portuguese (`student-por.csv`). This project must use **only `student-mat.csv`**. The CSV uses a semicolon (`;`) as its separator.

The code first checks whether the Mathematics CSV is already in `data/raw/`. If not, it downloads the official UCI ZIP archive and extracts the Mathematics file.

In [ ]:
DATA_FILE = DATA_DIR / "student-mat.csv"
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

def _extract_student_mat(archive, dest):
    """Search for student-mat.csv in a zip, including nested zips."""
    direct = [n for n in archive.namelist() if n.lower().endswith('student-mat.csv')]
    if direct:
        with archive.open(direct[0]) as src, open(dest, 'wb') as dst:
            dst.write(src.read())
        return True
    for name in archive.namelist():
        if name.lower().endswith('.zip'):
            with archive.open(name) as inner_file:
                with zipfile.ZipFile(BytesIO(inner_file.read())) as inner:
                    if _extract_student_mat(inner, dest):
                        return True
    return False

if not DATA_FILE.exists():
    print("Downloading the official UCI Student Performance archive...")
    response = requests.get(UCI_ZIP_URL, timeout=60)
    response.raise_for_status()

    with zipfile.ZipFile(BytesIO(response.content)) as archive:
        if not _extract_student_mat(archive, DATA_FILE):
            raise FileNotFoundError(
                "student-mat.csv was not found in the downloaded UCI ZIP. "
                "Download the archive manually from the UCI page and place student-mat.csv in data/raw/."
            )
    print("Mathematics CSV saved to:", DATA_FILE)
else:
    print("Using the existing Mathematics CSV:", DATA_FILE)

df = pd.read_csv(DATA_FILE, sep=";")

expected_columns = {"G1", "G2", "G3", "studytime", "absences", "failures"}
missing_expected = expected_columns.difference(df.columns)
if missing_expected:
    raise ValueError(f"The loaded file is not the expected Mathematics dataset. Missing columns: {sorted(missing_expected)}")

print("Dataset shape:", df.shape)
print("Expected Mathematics observations: about 395")
display(df.head())

## 3. Understand the data before modelling

Check column types, missing values, duplicate rows and summary statistics. This step is important because a model should not be trained before the dataset is understood.

In [ ]:
print("DATA TYPES")
display(df.dtypes.value_counts().rename_axis("Data type").to_frame("Number of columns"))

print("MISSING VALUES BY COLUMN")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values"))

print("Duplicate rows:", int(df.duplicated().sum()))

print("NUMERIC SUMMARY")
display(df.describe().T)

print("TARGET (G3) SUMMARY")
print(df[TARGET].describe())
print("Median:", df[TARGET].median())
print("Mode:", df[TARGET].mode().tolist())

### 3.1 Mean, median and standard deviation

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
summary_stats = pd.DataFrame({
    "mean": numeric_df.mean(),
    "median": numeric_df.median(),
    "standard_deviation": numeric_df.std()
})
display(summary_stats)

### 3.2 Covariance and correlation

Covariance and correlation describe how numeric variables vary together. Correlation is bounded between -1 and +1. These statistics show association, not cause and effect.

In [ ]:
print("COVARIANCE MATRIX")
display(numeric_df.cov())

print("CORRELATION WITH FINAL GRADE (G3)")
display(numeric_df.corr(numeric_only=True)[[TARGET]].sort_values(TARGET, ascending=False))

## 4. Exploratory data analysis (EDA)

The following plots help understand the target distribution, relationships between selected factors and final grades, and correlations among numeric variables.

In [ ]:
# 4.1 Distribution of final Mathematics grades
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x=TARGET, bins=15, kde=True)
plt.title("Distribution of Final Mathematics Grades (G3)")
plt.xlabel("Final grade (0 to 20)")
plt.ylabel("Number of students")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_target_distribution.png", dpi=150)
plt.show()

In [ ]:
# 4.2 Correlation heatmap for numeric variables
plt.figure(figsize=(14, 10))
corr = numeric_df.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.3)
plt.title("Correlation Heatmap - Numeric Variables")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "02_correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
# 4.3 Selected feature-versus-grade plots
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.boxplot(data=df, x="studytime", y=TARGET, ax=axes[0])
axes[0].set_title("Study Time vs Final Grade")
axes[0].set_xlabel("Weekly study-time category")

sns.scatterplot(data=df, x="absences", y=TARGET, alpha=0.7, ax=axes[1])
axes[1].set_title("Absences vs Final Grade")

sns.boxplot(data=df, x="failures", y=TARGET, ax=axes[2])
axes[2].set_title("Past Failures vs Final Grade")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "03_selected_features_vs_grade.png", dpi=150)
plt.show()

## 5. Prepare the predictors and target

- **Target (`y`):** `G3`, the final Mathematics grade.
- **Predictors (`X`):** all remaining columns except `G1` and `G2`.
- `G1` and `G2` are excluded by design so the model uses non-grade factors.

The train/test split is performed before fitting the preprocessing steps. The preprocessing is then placed inside a scikit-learn pipeline so imputers and encoders learn only from the training data. This helps prevent data leakage.

In [ ]:
X = df.drop(columns=["G1", "G2", TARGET])
y = df[TARGET].copy()

assert TARGET not in X.columns
assert "G1" not in X.columns and "G2" not in X.columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Number of predictors:", X.shape[1])
print("Numeric predictors:", len(numeric_features))
print("Categorical predictors:", len(categorical_features))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## 6. Build the preprocessing and Linear Regression pipeline

Numeric columns use median imputation. Categorical columns use most-frequent imputation followed by one-hot encoding. Even though the original UCI dataset is documented as having no missing values, the imputation steps make the workflow more robust to future input data.

In [ ]:
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", drop=None)
except TypeError:  # Compatibility fallback for older scikit-learn versions
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot_encoder)
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)
print("Linear Regression model training completed.")

## 7. Evaluate the model

This is a **regression** task, so do not report classification accuracy.

- **R-squared (R²):** measures the proportion of target variation explained by the model relative to a mean-only baseline. Test R² can be negative.
- **MAE:** average absolute prediction error in grade points.
- **RMSE:** square root of average squared error; larger errors receive more weight.

Compare training and testing results. A substantially better training score can indicate overfitting; weak scores on both sets can indicate that these predictors provide limited predictive information for this model.

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

def regression_metrics(actual, predicted):
    return {
        "R2": r2_score(actual, predicted),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted)))
    }

train_metrics = regression_metrics(y_train, y_train_pred)
test_metrics = regression_metrics(y_test, y_test_pred)
metrics_df = pd.DataFrame([train_metrics, test_metrics], index=["Train", "Test"])
metrics_df.index.name = "Dataset"
display(metrics_df.round(4))

metrics_df.to_csv(RESULTS_DIR / "evaluation_metrics.csv")

## 8. Visualise predictions and residuals

A residual is `actual grade - predicted grade`. Look for points around the diagonal in the actual-versus-predicted plot and for residuals scattered around zero without a strong pattern. These are diagnostic aids, not proof that a model is perfect.

In [ ]:
# 8.1 Actual versus predicted grades
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=y_test_pred, alpha=0.75)
low = min(float(y_test.min()), float(np.min(y_test_pred)))
high = max(float(y_test.max()), float(np.max(y_test_pred)))
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual G3")
plt.ylabel("Predicted G3")
plt.title("Actual vs Predicted Final Grades - Test Set")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "04_actual_vs_predicted.png", dpi=150)
plt.show()

In [ ]:
# 8.2 Residual plot
residuals = y_test - y_test_pred
plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_test_pred, y=residuals, alpha=0.75)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted G3")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residual Plot - Test Set")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_residual_plot.png", dpi=150)
plt.show()

## 9. Interpret model coefficients

A positive coefficient means that, holding the other model inputs constant, the feature is associated with a higher model prediction. A negative coefficient means an association with a lower prediction. Coefficients do not establish causation. The size of a coefficient also depends on the feature's units and encoding.

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["regressor"].coef_

coefficients_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})
coefficients_df["absolute_coefficient"] = coefficients_df["coefficient"].abs()
coefficients_df = coefficients_df.sort_values("absolute_coefficient", ascending=False)

display(coefficients_df.head(20).drop(columns="absolute_coefficient"))
coefficients_df.to_csv(RESULTS_DIR / "model_coefficients.csv", index=False)

plot_data = coefficients_df.head(20).sort_values("coefficient")
plt.figure(figsize=(10, 8))
plt.barh(plot_data["feature"], plot_data["coefficient"])
plt.axvline(0, linestyle="--")
plt.xlabel("Linear Regression coefficient")
plt.title("Largest 20 Coefficients by Absolute Magnitude")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_model_coefficients.png", dpi=150)
plt.show()

## 10. Save test predictions and the trained model

In [ ]:
predictions_df = pd.DataFrame({
    "actual_G3": y_test.reset_index(drop=True),
    "predicted_G3": pd.Series(y_test_pred).round(2),
    "residual": pd.Series(residuals.to_numpy()).round(2)
})
predictions_df.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

joblib.dump(model, RESULTS_DIR / "student_grade_linear_regression.joblib")

print("Saved results:")
for path in sorted(RESULTS_DIR.iterdir()):
    if path.is_file():
        print(" -", path.name)
print("Saved plots:")
for path in sorted(PLOTS_DIR.iterdir()):
    if path.is_file():
        print(" -", path.name)

## 11. Final interpretation

The following results were generated by running the notebook against the live UCI dataset.

### Model performance

| Metric | Training set | Testing set |
|--------|-------------|-------------|
| R²     | 0.2888      | 0.1415      |
| MAE    | 2.9462      | 3.3953      |
| RMSE   | 3.8651      | 4.1957      |

### Interpretation

1. **Training R² = 0.2888, Testing R² = 0.1415.** The model explains about 29% of the variance in final Mathematics grades on the training set and about 14% on the test set. These are modest values, indicating that the chosen non-grade features have limited predictive power for `G3` using a linear model.

2. **MAE ≈ 3.0–3.4 grade points.** On average, the model's prediction is off by about three grade points (on a 0–20 scale). The RMSE is slightly higher (~4.0–4.2), meaning some predictions have larger errors.

3. **Training vs. testing performance is reasonably close.** The gap between training and test R² (0.29 vs. 0.14) suggests some mild overfitting, but both scores are low. The model is not dramatically memorizing training data; rather, the features themselves provide limited linear signal for predicting `G3`.

4. **Strongest numeric correlations with G3 (excluding G1 and G2):**
   - `failures` has the strongest negative correlation (r ≈ −0.36): students with more past class failures tend to have lower final grades. This is an *association*, not proof that failures *cause* lower grades.
   - `Medu` (mother's education, r ≈ 0.22) and `Fedu` (father's education, r ≈ 0.15) show moderate positive associations.
   - `age` (r ≈ −0.16) and `goout` (going out with friends, r ≈ −0.13) show mild negative associations.

5. **Largest model coefficients:**
   - `failures` (−1.85): each additional past failure is associated with about 1.85 fewer predicted grade points.
   - `Mjob_teacher` (−1.19), `Mjob_health` (+1.04), `Fjob_teacher` (+0.89): mother's and father's occupation categories show notable associations.
   - `higher_yes` (+0.88): aspiration for higher education is associated with higher predicted grades.
   - `studytime` (+0.76): more weekly study time is associated with slightly higher predicted grades.

6. **Actual-versus-predicted plot:** Points are widely scattered around the diagonal, confirming the model's limited accuracy. The model tends to predict values closer to the mean and struggles with students who score very low (0–5) or very high (17–20).

7. **Residual plot:** Residuals are roughly centered around zero but show a wide spread (roughly ±10 grade points for some observations). There is a noticeable pattern: the model underpredicts for students with high actual grades and overpredicts for students with very low actual grades. This suggests the linear model fails to capture non-linear relationships.

### Limitations
- The data come from two Portuguese schools and may not represent other student populations.
- The data are observational; correlation does not establish causation.
- Linear Regression may miss nonlinear relationships.
- Results can change with the train/test split; cross-validation is a useful extension.
- A predicted grade is an estimate and should not be the sole basis for decisions about an individual student.

## 12. Dataset citation

Cortez, P. (2008). *Student Performance* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5TG7T

Official dataset page: https://archive.ics.uci.edu/dataset/320/student-performance